# TP — Les $k$ plus proches voisins : choisir un modèle sans se tromper

**Cours d'apprentissage supervisé — Chapitre 1 : Introduction**
Mohammed SEDKI — Université Paris-Saclay

---

**Objectifs.** Mettre en pratique les notions du chapitre sur des données biomédicales réelles :

1. constater que l'erreur d'entraînement est trompeuse ;
2. observer le compromis biais–variance à travers le choix de $k$ ;
3. choisir $k$ par **validation croisée**, sans jamais toucher à l'échantillon de test ;
4. comprendre le rôle de la standardisation et des **fuites d'information** ;
5. passer d'un classifieur à une **décision médicale** (sensibilité, spécificité, coûts) ;
6. observer le **fléau de la dimension** ;
7. traiter un problème de **régression**.

**Durée indicative :** 3 h. Les parties marquées ★ sont facultatives (pour aller plus loin).

**Données.** Deux jeux de données biomédicaux fournis avec `scikit-learn` (aucun téléchargement nécessaire) :
le diagnostic du cancer du sein (classification) et la progression du diabète (régression).

**Mode d'emploi.** Les cellules contenant `...` ou `# À COMPLÉTER` sont à compléter.
Les questions appellent une réponse rédigée dans la cellule « Votre réponse » qui suit.

## 0. Mise en route

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.model_selection import (train_test_split, StratifiedKFold, KFold,
                                     cross_val_score, GridSearchCV)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import confusion_matrix, mean_squared_error, r2_score
from sklearn.feature_selection import SelectKBest, f_classif

RANDOM_STATE = 0          # pour que tout le monde obtienne les mêmes résultats
plt.rcParams["figure.figsize"] = (7, 4)

## 1. Les données : diagnostic du cancer du sein (≈ 15 min)

Le jeu de données *Wisconsin Diagnostic Breast Cancer* décrit **569 tumeurs du sein**.
Pour chacune, un prélèvement par aspiration à l'aiguille fine a été photographié au microscope.
Sur l'image, on a mesuré 10 caractéristiques des noyaux cellulaires (rayon, texture, périmètre, aire,
régularité du contour, compacité, concavité, points concaves, symétrie, dimension fractale),
résumées de 3 façons (moyenne, erreur standard, pire valeur) : **$p = 30$ variables**.

La sortie est le diagnostic : tumeur **maligne** ou **bénigne**.

⚠️ Dans `scikit-learn`, la cible vaut 0 pour « maligne » et 1 pour « bénigne ». On la recode pour suivre la
convention médicale : **$y = 1$ pour maligne** (la classe « positive » est la maladie).

In [ ]:
donnees = load_breast_cancer(as_frame=True)
X = donnees.data                 # tableau 569 x 30
y = 1 - donnees.target           # 1 = maligne, 0 = bénigne

print("Dimensions de X :", X.shape)
print("Effectifs par classe :", y.value_counts().to_dict())

**Question 1.1.** Combien de tumeurs sont malignes ? Quel taux d'erreur obtiendrait un classifieur
qui répond toujours « bénigne », sans regarder $x$ ? Ce chiffre servira de **référence** : un modèle utile
doit faire nettement mieux.

In [ ]:
# À COMPLÉTER : proportion de tumeurs malignes
...

**Votre réponse :**

*…*

**Question 1.2.** Observez les ordres de grandeur de quelques variables. Les $k$-ppv utilisent la
distance euclidienne $\|x - x'\|$ : quelle variable va dominer le calcul des distances ? Qu'en concluez-vous ?

In [ ]:
X.describe().T.loc[["mean radius", "mean area", "mean smoothness", "worst area"],
                   ["mean", "std", "min", "max"]].round(3)

**Votre réponse :**

*…*

## 2. Le piège de l'erreur d'entraînement (≈ 20 min)

On met de côté **25 % des données comme échantillon de test**. Ce test ne servira qu'à l'évaluation finale.
L'option `stratify=y` garde la même proportion de tumeurs malignes dans les deux échantillons.

In [ ]:
X_app, X_test, y_app, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)

print("Apprentissage :", X_app.shape, "  Test :", X_test.shape)

def taux_erreur(modele, X, y):
    # Proportion de prédictions fausses (perte 0-1 moyenne).
    return np.mean(modele.predict(X) != y)

### 2.1 Le classifieur « par cœur »

On programme le classifieur du cours : il **mémorise** les couples $(x_i, y_i)$ d'apprentissage ;
face à un $x$ déjà vu, il renvoie le $y_i$ mémorisé, sinon il répond 0 (bénigne).

**Question 2.1.** Complétez la classe, puis calculez ses erreurs d'entraînement et de test.

In [ ]:
class ClassifieurParCoeur:
    def fit(self, X, y):
        # À COMPLÉTER : construire un dictionnaire  tuple(x_i) -> y_i
        self.memoire = ...
        return self

    def predict(self, X):
        # À COMPLÉTER : renvoyer la classe mémorisée si x a déjà été vu, 0 sinon
        # (indice : dict.get(cle, valeur_par_defaut))
        return np.array([... for x in np.asarray(X)])

par_coeur = ClassifieurParCoeur().fit(X_app, y_app)
print("Erreur d'entraînement :", round(taux_erreur(par_coeur, X_app, y_app), 3))
print("Erreur de test        :", round(taux_erreur(par_coeur, X_test, y_test), 3))

### 2.2 Le plus proche voisin

On utilise maintenant `KNeighborsClassifier(n_neighbors=1)`. On l'enchaîne avec une standardisation grâce à
`make_pipeline` : le *pipeline* calcule les moyennes et écarts-types **sur les données d'apprentissage**
puis les applique aux nouvelles données.

**Question 2.2.** Calculez les erreurs d'entraînement et de test du 1-ppv. Pourquoi les deux classifieurs
ont-ils une erreur d'entraînement nulle ? Pourquoi leurs erreurs de test sont-elles si différentes ?

In [ ]:
ppv1 = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=1))
# À COMPLÉTER : ajuster ppv1 sur l'échantillon d'apprentissage, puis afficher les deux erreurs
...

**Votre réponse :**

*…*

## 3. Influence de $k$ : la courbe en U (≈ 20 min)

**Question 3.1.** Pour $k = 1, \ldots, 60$, ajustez un $k$-ppv (avec standardisation) et stockez ses erreurs
d'entraînement et de test. Le code du graphique est fourni.

In [ ]:
valeurs_k = np.arange(1, 61)
err_app, err_test = [], []
for k in valeurs_k:
    # À COMPLÉTER : ajuster un k-ppv standardisé, puis ajouter ses erreurs aux deux listes
    ...

plt.plot(valeurs_k, err_app, "o-", ms=3, label="entraînement")
plt.plot(valeurs_k, err_test, "o-", ms=3, label="test")
plt.gca().invert_xaxis()          # complexité croissante vers la droite
plt.xlabel("k  (complexité croissante →)")
plt.ylabel("taux d'erreur")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

k_test = valeurs_k[np.argmin(err_test)]
print("k minimisant l'erreur de test :", k_test, " erreur :", round(min(err_test), 3))

**Question 3.2.** Décrivez les deux courbes. Où se situent le surapprentissage et le sous-apprentissage ?

**Question 3.3.** Un collègue propose de publier : « avec $k = 10$, notre méthode n'a que 3,5 % d'erreur ».
Qu'en pensez-vous ?

**Votre réponse :**

*…*

## 4. Un seul découpage ne suffit pas (≈ 15 min)

Et si l'on avait tiré un autre échantillon de test ? On répète le découpage apprentissage/test avec
30 graines aléatoires différentes. Pour chacune, on note le $k$ qui minimise l'erreur de test, et l'erreur
de test obtenue avec $k = 5$.

**Question 4.1.** Complétez la boucle, puis commentez les deux graphiques.

In [ ]:
meilleurs_k, err_k5 = [], []
for graine in range(30):
    Xa, Xt, ya, yt = train_test_split(X, y, test_size=0.25, stratify=y, random_state=graine)
    erreurs = []
    for k in valeurs_k:
        # À COMPLÉTER : ajuster un k-ppv standardisé sur (Xa, ya) et stocker son erreur sur (Xt, yt)
        ...
    meilleurs_k.append(int(valeurs_k[np.argmin(erreurs)]))
    err_k5.append(erreurs[4])                    # k = 5 est en position 4

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].hist(meilleurs_k, bins=np.arange(0.5, 25.5, 1))
axes[0].set_xlabel("k minimisant l'erreur de test"); axes[0].set_ylabel("nombre de découpages")
axes[1].hist(err_k5, bins=12)
axes[1].set_xlabel("erreur de test avec k = 5")
plt.tight_layout(); plt.show()

print("k choisis :", sorted(meilleurs_k))
print("Erreur de test pour k = 5 : de", round(min(err_k5), 3), "à", round(max(err_k5), 3))

**Votre réponse :**

*…*

## 5. Choisir $k$ par validation croisée (≈ 35 min)

Rappel du cours : on découpe l'échantillon d'apprentissage en $V$ blocs. Chaque bloc sert à tour de rôle de
bloc de validation ; le modèle est ajusté sur les $V - 1$ autres. L'erreur de validation croisée $\mathrm{CV}(k)$
est la moyenne des erreurs sur les blocs de validation.

**Le test reste de côté pendant toute cette partie.**

### 5.1 Programmer la validation croisée

`StratifiedKFold` fournit les indices des blocs (en gardant la proportion de malignes dans chaque bloc).

**Question 5.1.** Complétez la fonction `erreur_cv`.

In [ ]:
def erreur_cv(k, X, y, V=5):
    blocs = StratifiedKFold(n_splits=V, shuffle=True, random_state=RANDOM_STATE)
    erreurs = []
    for idx_app, idx_val in blocs.split(X, y):
        modele = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
        # À COMPLÉTER : ajuster le modèle SANS le bloc de validation,
        #               puis ajouter son erreur SUR le bloc de validation
        ...
    return np.mean(erreurs)

print("CV(1) =", round(erreur_cv(1, X_app, y_app), 3))

### 5.2 La courbe $\mathrm{CV}(k)$

**Question 5.2.** Calculez $\mathrm{CV}(k)$ pour $k = 1, \ldots, 60$ et déterminez $\hat k = \arg\min_k \mathrm{CV}(k)$.

In [ ]:
# À COMPLÉTER : liste des CV(k) pour k dans valeurs_k, puis k_chapeau
err_cv = ...
k_chapeau = ...

plt.plot(valeurs_k, err_cv, "o-", ms=3)
plt.axvline(k_chapeau, ls=":", color="gray")
plt.gca().invert_xaxis()
plt.xlabel("k"); plt.ylabel("erreur de validation croisée"); plt.grid(alpha=0.3)
plt.show()
print("k choisi par validation croisée :", k_chapeau, "  CV =", round(min(err_cv), 3))

### 5.3 La même chose avec `scikit-learn`

`cross_val_score` calcule les scores sur les blocs (par défaut le taux de bonnes réponses, donc
erreur $= 1 -$ score). `GridSearchCV` fait toute la boucle sur $k$ et **réajuste automatiquement** le modèle
retenu sur tout l'échantillon d'apprentissage.

**Question 5.3.** Vérifiez que vous retrouvez la même erreur et le même $\hat k$.

In [ ]:
blocs = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# À COMPLÉTER : cross_val_score pour le modèle avec k = k_chapeau
scores = ...
print("cross_val_score : CV =", round(1 - scores.mean(), 3))

# Le nom du paramètre suit la règle  <nom de l'étape en minuscules>__<paramètre>
grille = {"kneighborsclassifier__n_neighbors": valeurs_k}
# À COMPLÉTER : GridSearchCV sur le pipeline standardisation + k-ppv, puis .fit(...)
recherche = ...
print("GridSearchCV    : k =", recherche.best_params_["kneighborsclassifier__n_neighbors"],
      " CV =", round(1 - recherche.best_score_, 3))

### 5.4 Évaluation finale, une seule fois

**Question 5.4.** Évaluez le modèle retenu (`recherche`, déjà réajusté) sur l'échantillon de test.
L'erreur de test est-elle égale à l'erreur de validation croisée ? Pourquoi ?
*Indication :* l'écart-type d'une proportion d'erreurs $\hat p$ estimée sur $m$ individus est environ
$\sqrt{\hat p(1 - \hat p)/m}$.

In [ ]:
# À COMPLÉTER : erreur de test du modèle retenu et son écart-type approximatif
err_finale = ...
m = len(y_test)
ecart_type = ...
print("Erreur de test :", round(err_finale, 3))
print("Écart-type approximatif :", round(ecart_type, 3))

**Votre réponse :**

*…*

## 6. Standardiser ou non ? (≈ 15 min)

**Question 6.1.** Calculez la courbe $\mathrm{CV}(k)$ **sans** standardisation et comparez-la à celle de la partie 5.

In [ ]:
# À COMPLÉTER : CV(k) pour un k-ppv SANS standardisation
err_cv_brut = ...

plt.plot(valeurs_k, err_cv_brut, "o-", ms=3, label="sans standardisation")
plt.plot(valeurs_k, err_cv, "o-", ms=3, label="avec standardisation")
plt.gca().invert_xaxis()
plt.xlabel("k"); plt.ylabel("erreur de validation croisée"); plt.legend(); plt.grid(alpha=0.3)
plt.show()
print("Meilleure CV sans standardisation :", round(min(err_cv_brut), 3))
print("Meilleure CV avec standardisation :", round(min(err_cv), 3))

**Question 6.2.** Pourquoi la standardisation est-elle placée **dans** le pipeline plutôt qu'appliquée
une fois pour toutes à `X` avant le découpage ?

**Votre réponse :**

*…*

## 7. Du vote à la décision médicale (≈ 20 min)

### 7.1 Matrice de confusion

On note « positif » une tumeur prédite maligne. Deux indicateurs sont utilisés en médecine :

* **sensibilité** $= \dfrac{VP}{VP + FN}$ : proportion de tumeurs malignes détectées ;
* **spécificité** $= \dfrac{VN}{VN + FP}$ : proportion de tumeurs bénignes reconnues comme telles.

**Question 7.1.** Calculez la matrice de confusion du modèle retenu sur le test, puis sa sensibilité et sa
spécificité.

In [ ]:
def bilan(y_vrai, y_pred):
    vn, fp, fn, vp = confusion_matrix(y_vrai, y_pred).ravel()
    print(f"VN = {vn}, FP = {fp}, FN = {fn}, VP = {vp}")
    # À COMPLÉTER : afficher la sensibilité et la spécificité
    ...

bilan(y_test, recherche.predict(X_test))

### 7.2 Changer le seuil selon les coûts

`predict_proba` renvoie, pour chaque patient, la proportion de tumeurs malignes parmi ses $k$ voisins :
c'est l'estimation $\hat\eta_k(x)$ de $\eta(x) = \P(Y = 1 \mid X = x)$. Par défaut, on prédit « maligne » si
$\hat\eta_k(x) > 1/2$.

Le cours montre qu'avec des coûts asymétriques, le seuil optimal est $\dfrac{c_{FP}}{c_{FP} + c_{FN}}$.

**Question 7.2.** On estime que rater un cancer (faux négatif) coûte 5 fois plus cher qu'une biopsie inutile
(faux positif) : $c_{FN} = 5$, $c_{FP} = 1$. Calculez le seuil, appliquez-le et comparez les deux bilans.

In [ ]:
eta_chapeau = recherche.predict_proba(X_test)[:, 1]      # proportion de voisins malins
print("Valeurs prises par eta_chapeau :", np.unique(eta_chapeau))

c_FN, c_FP = 5, 1
# À COMPLÉTER : seuil optimal, puis bilans pour le seuil 1/2 et pour ce seuil
seuil = ...
...

**Votre réponse :**

*…*

## 8. ★ Le fléau de la dimension (≈ 20 min)

On ajoute aux 30 variables $q$ variables de **pur bruit** (tirées au hasard, sans lien avec le diagnostic),
comme dans une étude où l'on mesure beaucoup de choses inutiles. On compare l'erreur de validation croisée
du $k$-ppv ($k$ choisi par `GridSearchCV`) à celle d'une **régression logistique** : un modèle linéaire pour la
classification, vu plus loin dans le cours, utilisé ici comme une boîte noire.

**Question 8.1.** Lancez la cellule et commentez.

In [ ]:
generateur = np.random.default_rng(RANDOM_STATE)
grille_k = {"kneighborsclassifier__n_neighbors": np.arange(1, 61, 2)}
resultats = []
for q in [0, 10, 30, 100, 300, 1000]:
    bruit = generateur.standard_normal((len(X_app), q))
    X_bruite = np.hstack([X_app.values, bruit])
    kppv = GridSearchCV(make_pipeline(StandardScaler(), KNeighborsClassifier()), grille_k, cv=blocs)
    kppv.fit(X_bruite, y_app)
    logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
    err_logit = 1 - cross_val_score(logit, X_bruite, y_app, cv=blocs).mean()
    resultats.append({"variables de bruit": q, "k-ppv": 1 - kppv.best_score_, "logistique": err_logit})

resultats = pd.DataFrame(resultats).set_index("variables de bruit")
resultats.plot(marker="o")
plt.xscale("symlog", linthresh=10)         # échelle adaptée à q = 0, 10, ..., 1000
plt.ylabel("erreur de validation croisée"); plt.grid(alpha=0.3); plt.show()
resultats.round(3)

**Votre réponse :**

*…*

## 9. ★ Données omiques : la mauvaise façon de faire une validation croisée (≈ 15 min)

Situation typique en génomique : **$n = 50$ patients, $p = 5000$ gènes**. Ici, les « expressions de gènes » sont
du **pur bruit** et les classes sont tirées au hasard : aucune méthode ne peut faire mieux que **50 %**
d'erreur.

Une pratique répandue consiste à **sélectionner d'abord** les 100 gènes les plus liés à $y$ sur toutes
les données, puis à faire la validation croisée sur ces 100 gènes.

**Question 9.1.** Comparez cette « mauvaise » validation croisée à la « bonne », où la sélection est faite
**dans** le pipeline, donc recalculée dans chaque bloc. On répète l'expérience sur 20 jeux de données.

In [ ]:
erreurs_mauvaise, erreurs_bonne = [], []
for rep in range(20):
    gen = np.random.default_rng(rep)
    X_omique = gen.standard_normal((50, 5000))          # 50 patients, 5000 gènes (bruit)
    y_omique = np.repeat([0, 1], 25)                      # classes sans lien avec X
    blocs_omique = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    # Mauvaise façon : sélection sur TOUTES les données, puis validation croisée
    X_selection = SelectKBest(f_classif, k=100).fit_transform(X_omique, y_omique)
    erreurs_mauvaise.append(1 - cross_val_score(KNeighborsClassifier(n_neighbors=1),
                                                X_selection, y_omique, cv=blocs_omique).mean())

    # Bonne façon : À COMPLÉTER (pipeline sélection + 1-ppv, évalué par validation croisée)
    modele = ...
    erreurs_bonne.append(...)

print("Erreur CV, mauvaise façon :", round(np.mean(erreurs_mauvaise), 3))
print("Erreur CV, bonne façon    :", round(np.mean(erreurs_bonne), 3))

**Votre réponse :**

*…*

## 10. Régression : progression du diabète (≈ 25 min)

Le jeu de données *diabetes* décrit **442 patients diabétiques** : âge, sexe, indice de masse corporelle (`bmi`),
pression artérielle moyenne (`bp`) et 6 mesures sanguines (`s1` à `s6`). La sortie $y$ est une **mesure
quantitative de la progression de la maladie un an après**. Les variables sont déjà centrées et mises à
l'échelle par `scikit-learn`.

On utilise la perte quadratique : l'erreur est l'**erreur quadratique moyenne** (EQM).

In [ ]:
Xd, yd = load_diabetes(return_X_y=True, as_frame=True)
Xd_app, Xd_test, yd_app, yd_test = train_test_split(Xd, yd, test_size=0.25, random_state=RANDOM_STATE)
print("Apprentissage :", Xd_app.shape, "  Test :", Xd_test.shape)
Xd.head()

**Question 10.1.** Référence naïve : d'après le cours, quelle est la meilleure prédiction **constante**
pour la perte quadratique ? Calculez son EQM de test.

In [ ]:
# À COMPLÉTER : prédire la moyenne de yd_app pour tous les patients du test
constante = ...
eqm_constante = ...
print("Prédiction constante :", round(constante, 1), "  EQM de test :", round(eqm_constante, 1))

**Question 10.2.** Choisissez $k$ par validation croisée à 5 blocs pour un `KNeighborsRegressor`
(`scoring="neg_mean_squared_error"` : `scikit-learn` maximise les scores, d'où le signe moins).
Tracez la courbe de validation croisée et calculez l'EQM de test du modèle retenu.

In [ ]:
blocs_reg = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
grille_reg = {"kneighborsregressor__n_neighbors": valeurs_k}
# À COMPLÉTER : GridSearchCV pour standardisation + KNeighborsRegressor, puis .fit(...)
kppv_reg = ...

plt.plot(valeurs_k, -kppv_reg.cv_results_["mean_test_score"], "o-", ms=3)
plt.gca().invert_xaxis()
plt.xlabel("k"); plt.ylabel("EQM de validation croisée"); plt.grid(alpha=0.3)
plt.show()

eqm_kppv = ...
print("k =", kppv_reg.best_params_["kneighborsregressor__n_neighbors"],
      " EQM CV :", round(-kppv_reg.best_score_, 1), " EQM test :", round(eqm_kppv, 1))

**Question 10.3.** Comparez avec une **régression linéaire** (moindres carrés), puis complétez le tableau.
Le coefficient $R^2 = 1 - \text{EQM} / \text{variance de } y$ mesure la part de variabilité expliquée.
Quelle méthode retenir ici ? Est-ce en contradiction avec la partie 8 ?

In [ ]:
# À COMPLÉTER : ajuster une régression linéaire et calculer son EQM de test
lineaire = ...
eqm_lineaire = ...

bilan_reg = pd.DataFrame({
    "EQM de test": [eqm_constante, eqm_kppv, eqm_lineaire],
    "R2 de test": [r2_score(yd_test, np.full(len(yd_test), constante)),
                   r2_score(yd_test, kppv_reg.predict(Xd_test)),
                   r2_score(yd_test, lineaire.predict(Xd_test))]},
    index=["constante (moyenne)", f"k-ppv (k = {kppv_reg.best_params_['kneighborsregressor__n_neighbors']})",
           "régression linéaire"])
bilan_reg.round(3)

**Votre réponse :**

*…*

## Bilan : les bonnes pratiques

1. **Mettre de côté un échantillon de test** dès le début, et ne l'utiliser qu'une fois, à la fin.
2. **Ne jamais juger un modèle sur son erreur d'entraînement** (parties 2 et 3).
3. **Choisir les hyperparamètres** (ici $k$) **par validation croisée** sur l'échantillon d'apprentissage (partie 5).
4. **Mettre toute transformation dans le pipeline** : standardisation, sélection de variables… (parties 6 et 9).
5. **Toujours comparer à une référence naïve** (classe majoritaire, moyenne).
6. **Annoncer l'erreur avec son incertitude**, et **choisir le seuil selon les coûts** du problème (partie 7).
7. **Aucune méthode n'est la meilleure partout** : comparer plusieurs familles de modèles (parties 8 et 10).